# 08 Freeze and independent evaluation

## Learning outcome

Verify which artifacts are frozen, describe the blind entry point, and distinguish missing files from failed evaluation.

## Method and notation

The freeze records data and code hashes, checkpoint hashes, score bounds and the diagnostic convention. The blind command loads only frozen predictors and checks exact numerical overlap. Actual acquisition identity requires separate metadata. A missing test file creates an explicit status record rather than an empty score presented as evidence.

Prerequisites are listed in the project README. This notebook reads the configured run directory. The tables and measurements below come from the recorded run or are recomputed in Python. Use `scripts/execute_notebooks.py --mode full --run-id NEW --data-dir PATH` to regenerate the shared pipeline.

In [ ]:
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Image, display
from gearbox_dhon.data import find_input, load_record

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
run_id = os.getenv('GEARBOX_RUN_ID', 'development')
run = root / 'outputs' / 'runs' / run_id
candidate = Path(os.getenv('GEARBOX_DATA_DIR', str(root / 'data' / 'raw')))
data_dir = candidate if candidate.is_dir() and find_input(candidate, 'sigtrain') else None

def read(relative):
    path = run / relative
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

def show_files(directory, patterns, limit):
    candidates = [p for p in sorted(directory.rglob('*'))
                  if p.suffix in {'.csv', '.json'} and any(k in p.stem.lower() for k in patterns)]
    for path in candidates[:limit]:
        print(path.relative_to(root))
        if path.suffix == '.csv':
            display(pd.read_csv(path).head(8))
        else:
            value = json.loads(path.read_text())
            print(json.dumps(value, indent=2)[:3600])

def show_figures(patterns, limit):
    candidates = [p for p in sorted((run / 'figures').rglob('*.png'))
                  if any(k in p.stem.lower() for k in patterns)]
    for path in candidates[:limit]:
        print(path.relative_to(root))
        display(Image(filename=str(path), width=690))


## Recorded experiment

Inspect the measurements below against the stated information cutoff and split.

In [ ]:
manifest = json.loads((run / 'frozen_manifest.json').read_text())
print('Frozen models:', ', '.join(manifest['models']))
print('Prediction mask starts at:', manifest['scoring_start_zero_based'])
print('Threshold policy:', manifest['diagnostic_policy'])
for path in sorted(run.parent.glob('*/data_availability.json')):
    if path.parent != run:
        print(path.parent.name, json.loads(path.read_text()))
print('For future independent evaluation see the hash-checked command in README.md.')


## Interpretation

Review data scope and limitations in the [course report](../docs/final_course_report.md) and [companion guide](../docs/companion_guide.md). The executed values are attributable to the named run and the exact input and checkpoint hashes in its manifest.